# One-layer attention-only transformer

This time, we allow for a single layer of attention, with multiple heads.

We'll start with the training data. So we train on a mix of three sources, each kept as its own token stream (`src/mixed.py`):

- **wiki**: WikiText-103 (~130M tokens)
- **code**: Python files from GitHub, `codeparrot/codeparrot-clean` (~185M tokens)
- **math**: `hoskinson-center/proof-pile`, mostly arXiv LaTeX plus math StackExchange and formal proofs (~226M tokens)

One 16k byte-level BPE is trained on an equal sample of all three, so no source dominates the merges. Its pre-tokenizer keeps LaTeX commands whole (`\left`, ` \frac`), which plain GPT-2 splitting would break into `\` + `left`.

Set `SMOKE=1` in the environment before starting the kernel to run every cell in about a minute: training shrinks to a few steps and writes a separate checkpoint, so it never touches the real one.

In [1]:
import math
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
from src.mixed import REPO_ROOT, SOURCES, load_mixed

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
SMOKE = os.environ.get("SMOKE") == "1"

corpus = load_mixed(vocab_size=16384)
V = corpus.vocab_size

# streams live on the device as int32 (~2.2 GB for all of train) and are cast to int64 per batch
train_t = {s: torch.from_numpy(ids).to(device) for s, ids in corpus.train.items()}
val_t = {s: torch.from_numpy(ids).to(device) for s, ids in corpus.val.items()}
for s in SOURCES:
    print(f"{s:>4}: train {len(train_t[s]) / 1e6:6.1f}M, val {len(val_t[s]) / 1e6:4.1f}M tokens")
print(f"vocab {V:,}, device {device}{', SMOKE run' if SMOKE else ''}")

wiki: train  130.6M, val  0.3M tokens
code: train  184.8M, val  3.8M tokens
math: train  226.3M, val  6.5M tokens
vocab 16,384, device cuda


We'll once again need the bigram statistics to verify the claim that the $\text{Id} \otimes W_U W_E$ term contributes bigram statistics to the output. We retrieve them by again checking the `[t, t+1]` pairs.

In [2]:
# count every [t, t+1] pair: pair (a, b) maps to bin a * vocab + b. chunked so the int64 keys stay small
@torch.no_grad()
def count_bigrams(ids, chunk=50_000_000):
    counts = torch.zeros(V * V, dtype=torch.long, device=ids.device)
    for i in range(0, len(ids) - 1, chunk):
        a = ids[i : i + chunk + 1].long()
        counts += torch.bincount(a[:-1] * V + a[1:], minlength=V * V)
    return counts.view(V, V)

# kept as float32 (1 GB at 16k vocab) rather than int64 (2 GB)
pairwise_counts = torch.zeros(V, V, device=device)
for ids in train_t.values():
    pairwise_counts += count_bigrams(ids).float()

# bigram frequencies = pairwise_counts row-wise normalized, shown for a few tokens from each domain
for word in [" the", " import", "\\left", "\\begin", " def"]:
    i = corpus.encode(word)[0]
    row = pairwise_counts[i] / pairwise_counts[i].sum().clamp(min=1)
    top = row.topk(5)
    print(f"{corpus.token_str(i)!r:>10} → " + ", ".join(f"{corpus.token_str(j)!r} {p:.2f}" for p, j in zip(top.values.tolist(), top.indices.tolist())))

    ' the' → ' first' 0.01, ' following' 0.01, ' same' 0.01, '\n' 0.01, ' $' 0.01
 ' import' → ' *' 0.03, ' (' 0.03, '_' 0.02, ' get' 0.02, ' absolute' 0.02
  '\\left' → '(' 0.61, '[' 0.09, '\\{' 0.08, '|' 0.05, '\\vert' 0.02
 '\\begin' → '{' 1.00, ' {' 0.00, '_' 0.00, 'picture' 0.00, 'a' 0.00
    ' def' → ' test' 0.21, ' __' 0.13, ' _' 0.10, ' get' 0.06, ' set' 0.02


Now the 1-layer attention-only transformer. The class has two forward passes over the same weights:

- `forward_per_head` is the walkthrough: one head at a time, one column per token, every intermediate (scores, pattern, head output) spelled out, the way the paper breaks the model down. It is slow and memory-hungry at 1024 tokens (it stores a 1024×1024 pattern per head) but it is the only path that can return attention patterns.
- `forward` computes the same function with PyTorch's fused attention kernel (`scaled_dot_product_attention`) and is what training uses.

The fused kernel raises two questions, answered below: is it really the same model (so the QK/OV circuits read off the weights describe what was trained), and how do we look at attention patterns when it never materializes them? The equivalence check runs before training, so a mismatch stops Run All before the long run.

In [3]:
class OneLayerAttentionOnlyTransformer(nn.Module):
    def __init__(self, vocab_dim: int, hidden_dim: int, num_heads: int, head_dim: int, device: str):
        super().__init__()

        self.vocab_dim = vocab_dim
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.device = device

        # init only: each weight is scaled by 1/sqrt(fan_in) so activations stay O(1) through every matmul, and W_O
        # also by 1/sqrt(num_heads) so the heads' sum doesn't swamp the residual stream. the forward pass has no
        # LayerNorm or biases, so the QK/OV circuit expressions stay exact
        self.W_E = nn.Parameter(torch.randn(vocab_dim, hidden_dim, device=device))
        self.W_U = nn.Parameter(torch.randn(hidden_dim, vocab_dim, device=device) / hidden_dim ** 0.5)

        # attention
        self.W_Q = nn.Parameter(torch.randn(num_heads, hidden_dim, head_dim, device=device) / hidden_dim ** 0.5)
        self.W_K = nn.Parameter(torch.randn(num_heads, hidden_dim, head_dim, device=device) / hidden_dim ** 0.5)
        self.W_V = nn.Parameter(torch.randn(num_heads, hidden_dim, head_dim, device=device) / hidden_dim ** 0.5)
        self.W_O = nn.Parameter(torch.randn(num_heads, head_dim, hidden_dim, device=device) / (head_dim * num_heads) ** 0.5)

    # the walkthrough: as noted in the paper, not the most efficient way to do this, but each step is visible.
    # x is token ids, one window (T,) or a batch (B, T). activations keep one column per token, so a batch just adds
    # a leading dim; .mT transposes the last two dims and the weight matmuls broadcast over the batch
    def forward_per_head(self, x, return_patterns=False):
        # embed the token: (..., hidden, T)
        x0 = self.W_E[x].mT
        n = x0.shape[-1]
        mask = torch.triu(torch.ones(n, n, dtype=torch.bool, device=x0.device), 1)

        # each attention head is run and added to residual stream; note each head runs on the previous
        # residual and NOT the one currently being formed
        x1 = x0.clone()
        patterns = []
        for h in range(self.num_heads):
            q = self.W_Q[h].T @ x0
            k = self.W_K[h].T @ x0
            v = self.W_V[h].T @ x0
            scores = q.mT @ k / self.head_dim ** 0.5
            A = scores.masked_fill(mask, float("-inf")).softmax(-1)   # A[i, j]: how much token i attends to token j
            z = v @ A.mT
            x1 = x1 + self.W_O[h].T @ z
            patterns.append(A)

        # unembed: (..., T, vocab)
        logits = (self.W_U.T @ x1).mT
        return (logits, torch.stack(patterns, dim=-3)) if return_patterns else logits   # patterns: (..., heads, T, T)

    # the same function, all heads at once. activations here are one ROW per token, the transpose of the walkthrough,
    # which is the layout the fused kernel expects. scaled_dot_product_attention applies the same 1/sqrt(head_dim)
    # scale, and is_causal=True is the same mask (a token may attend to itself), but it never materializes the
    # (T, T) patterns, so it can't return them
    def forward(self, x):
        x0 = self.W_E[x]                                                  # (..., T, hidden)
        q = torch.einsum("...td,hde->...hte", x0, self.W_Q)               # (..., heads, T, head_dim)
        k = torch.einsum("...td,hde->...hte", x0, self.W_K)
        v = torch.einsum("...td,hde->...hte", x0, self.W_V)
        z = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x1 = x0 + torch.einsum("...hte,hed->...td", z, self.W_O)          # sum of every head's output, added once
        return x1 @ self.W_U                                              # (..., T, vocab)

H_DIM = 1024
NUM_HEADS = 32
HEAD_DIM = 128
CONTEXT_LEN = 1024

toy_transformer = OneLayerAttentionOnlyTransformer(V, hidden_dim=H_DIM, num_heads=NUM_HEADS, head_dim=HEAD_DIM, device=device)
print(f"{sum(p.numel() for p in toy_transformer.parameters()) / 1e6:.1f}M parameters")

# the fused path must compute the same function as the walkthrough, in full precision, or everything read off the
# weights later describes a different model. rel_err is the largest logit difference relative to the largest logit
@torch.no_grad()
def check_fused_matches_walkthrough(model, x):
    ref, patterns = model.forward_per_head(x, return_patterns=True)
    fused = model(x)
    rel_err = ((fused - ref).abs().max() / ref.abs().max()).item()
    assert rel_err < 1e-4, f"fused forward differs from the walkthrough: relative error {rel_err:.2e}"
    # training runs under bf16 autocast, which rounds activations to ~3 significant digits; this is the size of that
    # rounding on the same input, for scale (it changes the numbers the model trains on, not the function it computes)
    with torch.autocast(device, dtype=torch.bfloat16):
        bf16_err = ((model(x).float() - ref).abs().max() / ref.abs().max()).item()
    return rel_err, bf16_err

x_check = train_t["code"][: 2 * 256].long().view(2, 256)
rel_err, bf16_err = check_fused_matches_walkthrough(toy_transformer, x_check)
print(f"fused vs walkthrough (fp32): relative error {rel_err:.1e}; under bf16 autocast: {bf16_err:.1e}")

50.3M parameters
fused vs walkthrough (fp32): relative error 5.7e-07; under bf16 autocast: 5.0e-03


Training uses the fused forward under bf16 autocast (weights and optimizer state stay in fp32). Each batch draws `BATCH_SIZE` random 1024-token windows, split across sources by `MIX`, so a window never crosses from one domain to another and val loss can be read per domain. With no LayerNorm there is only one shot at stability, so the learning rate warms up and then follows a cosine decay, gradients are clipped, and a non-finite loss stops the run.

The run checkpoints to `checkpoints/`. If the final checkpoint exists, the cell loads it and skips training, so re-running the notebook never retrains; if only a partial one exists (a crash mid-run), it resumes from there.

In [4]:
BATCH_SIZE = 32          # windows per forward pass; 32 * 1024 tokens fits in ~9 GB with bf16 and the fused kernel
ACCUM_STEPS = 1          # batches per optimizer step
TRAIN_TOKENS = 400_000_000
MIX = {"wiki": 0.4, "code": 0.3, "math": 0.3}   # share of each batch's windows drawn from each source
PEAK_LR = 1e-3
WARMUP_STEPS = 500
GRAD_CLIP = 1.0
EVAL_EVERY = 500
VAL_WINDOWS = 64         # fixed val windows per source
CKPT_EVERY = 2000

CKPT = REPO_ROOT / "checkpoints" / f"1l_attn_mixed{'_smoke' if SMOKE else ''}.pt"
CKPT_PARTIAL = CKPT.with_suffix(".partial.pt")
if SMOKE:
    TRAIN_TOKENS, WARMUP_STEPS, EVAL_EVERY, CKPT_EVERY = 20 * BATCH_SIZE * CONTEXT_LEN, 5, 10, 10

TRAIN_STEPS = TRAIN_TOKENS // (ACCUM_STEPS * BATCH_SIZE * CONTEXT_LEN)
config = dict(vocab_size=V, hidden_dim=H_DIM, num_heads=NUM_HEADS, head_dim=HEAD_DIM, context_len=CONTEXT_LEN,
              train_tokens=TRAIN_TOKENS, mix=MIX, tokenizer=f"data/mixed/bpe-mixed-{V}.json")

# random windows from one stream, as (n, T) inputs and targets
def windows(ids, starts):
    chunk = ids[starts[:, None] + torch.arange(CONTEXT_LEN + 1, device=ids.device)].long()
    return chunk[:, :-1], chunk[:, 1:]

def sample_batch(streams, n):
    per_source = torch.multinomial(torch.tensor(list(MIX.values())), n, replacement=True).bincount(minlength=len(MIX))
    xs, ys = [], []
    for s, k in zip(MIX, per_source.tolist()):
        if k:
            x, y = windows(streams[s], torch.randint(len(streams[s]) - CONTEXT_LEN - 1, (k,), device=device))
            xs.append(x); ys.append(y)
    return torch.cat(xs), torch.cat(ys)

g = torch.Generator(device=device).manual_seed(0)
val_starts = {s: torch.randint(len(ids) - CONTEXT_LEN - 1, (VAL_WINDOWS,), device=device, generator=g) for s, ids in val_t.items()}

@torch.no_grad()
def val_loss(model, s):
    x, y = windows(val_t[s], val_starts[s])
    with torch.autocast(device, dtype=torch.bfloat16):
        return sum(F.cross_entropy(model(xb).float().flatten(0, 1), yb.flatten()).item()
                   for xb, yb in zip(x.split(BATCH_SIZE), y.split(BATCH_SIZE))) / len(x.split(BATCH_SIZE))

# reference point per source: predicting every token from its train frequency alone (add-one smoothed)
unigram = sum(torch.bincount(ids, minlength=V) for ids in train_t.values()).float() + 1
unigram_logp = (unigram / unigram.sum()).log()
unigram_val = {s: -unigram_logp[ids.long()].mean().item() for s, ids in val_t.items()}

# linear warmup to PEAK_LR, then cosine decay to 10% of it
def lr_at(step):
    if step < WARMUP_STEPS:
        return PEAK_LR * (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, TRAIN_STEPS - WARMUP_STEPS)
    return PEAK_LR * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress)))

opt = torch.optim.Adam(toy_transformer.parameters(), lr=PEAK_LR)
history = []   # (step, train loss, {source: val loss})

def save(path, step):
    path.parent.mkdir(exist_ok=True)
    torch.save(dict(model=toy_transformer.state_dict(), opt=opt.state_dict(), step=step, config=config, history=history), path)

start = 0
if CKPT.exists():
    ckpt = torch.load(CKPT, map_location=device)
    toy_transformer.load_state_dict(ckpt["model"])
    history, start = ckpt["history"], TRAIN_STEPS
    print(f"loaded {CKPT.name} (trained {ckpt['step']} steps); skipping training")
elif CKPT_PARTIAL.exists():
    ckpt = torch.load(CKPT_PARTIAL, map_location=device)
    toy_transformer.load_state_dict(ckpt["model"])
    opt.load_state_dict(ckpt["opt"])
    history, start = ckpt["history"], ckpt["step"]
    print(f"resuming from {CKPT_PARTIAL.name} at step {start}")

print(f"{TRAIN_STEPS:,} steps of {ACCUM_STEPS * BATCH_SIZE * CONTEXT_LEN:,} tokens; unigram val loss "
      + ", ".join(f"{s} {l:.2f}" for s, l in unigram_val.items()))
recent = []
for step in tqdm(range(start, TRAIN_STEPS), desc="training...", initial=start, total=TRAIN_STEPS):
    for group in opt.param_groups:
        group["lr"] = lr_at(step)
    opt.zero_grad()
    step_loss = 0.0
    for _ in range(ACCUM_STEPS):
        x, y = sample_batch(train_t, BATCH_SIZE)
        with torch.autocast(device, dtype=torch.bfloat16):
            logits = toy_transformer(x)
        # loss in fp32; scale by 1/ACCUM_STEPS so accumulated grads are the mean over the step's batches
        loss = F.cross_entropy(logits.float().flatten(0, 1), y.flatten()) / ACCUM_STEPS
        loss.backward()
        step_loss += loss.item()
    if not math.isfinite(step_loss):
        raise RuntimeError(f"loss is {step_loss} at step {step}; last partial checkpoint: {CKPT_PARTIAL}")
    torch.nn.utils.clip_grad_norm_(toy_transformer.parameters(), GRAD_CLIP)
    opt.step()
    recent.append(step_loss)
    if (step + 1) % EVAL_EVERY == 0 or step + 1 == TRAIN_STEPS:
        vals = {s: val_loss(toy_transformer, s) for s in SOURCES}
        history.append((step + 1, sum(recent) / len(recent), vals))
        recent = []
        tqdm.write(f"step {step + 1}: train {history[-1][1]:.2f}, val " + ", ".join(f"{s} {l:.2f}" for s, l in vals.items()))
    if (step + 1) % CKPT_EVERY == 0:
        save(CKPT_PARTIAL, step + 1)

if start < TRAIN_STEPS:
    save(CKPT, TRAIN_STEPS)
    CKPT_PARTIAL.unlink(missing_ok=True)
    print(f"saved {CKPT}")
toy_transformer.eval()

12,207 steps of 32,768 tokens; unigram val loss wiki 8.06, code 7.39, math 6.96


training...:   4%|▍         | 501/12207 [00:47<23:27,  8.32it/s]

step 500: train 6.28, val wiki 5.86, code 5.37, math 5.03


training...:   8%|▊         | 1001/12207 [01:34<22:38,  8.25it/s]

step 1000: train 5.19, val wiki 5.47, code 5.04, math 4.71


...

training...:  98%|█████████▊| 11999/12207 [18:50<00:19, 10.63it/s]

step 12000: train 4.05, val wiki 4.64, code 3.88, math 3.92


training...: 100%|██████████| 12207/12207 [19:11<00:00, 10.60it/s]


step 12207: train 4.05, val wiki 4.64, code 3.87, math 3.91
saved /home/saafetensors/Workspace/toy-transformers/checkpoints/1l_attn_mixed.pt


OneLayerAttentionOnlyTransformer()

The trained weights go back through both paths once more: the fused forward must still match the walkthrough on real text. Then, since the fused kernel never forms attention patterns, we get them from `forward_per_head` on a short snippet and show where each head's last token looks. This is the walkthrough doing its job: the same weights, one head at a time.

In [5]:
SNIPPETS = {
    "code": "import numpy as np\nimport torch.nn as nn\n\n\nclass Encoder(nn.Module):\n    def __init__(self, dim):\n        super().__init__()\n        self.dim = dim\n\n    def forward(self, x):\n        return np",
    "math": r"By the triangle inequality, $\left| \frac{a}{b} - c \right| \leq \epsilon$, so \begin{equation} \sum_{i=1}^{n} x_i \end",
}
TOP_ATTENDED = 3

for name, text in SNIPPETS.items():
    x = torch.tensor(corpus.encode(text), device=device)
    rel_err, _ = check_fused_matches_walkthrough(toy_transformer, x)
    _, patterns = toy_transformer.forward_per_head(x, return_patterns=True)   # (heads, T, T)
    print(f"\n{name}: ...{text[-50:]!r}  (fused vs walkthrough on trained weights: {rel_err:.1e})")
    print(f"where the last token {corpus.token_str(x[-1])!r} attends, per head:")
    last = patterns[:, -1, :]                                                   # (heads, T) over source positions
    for h in range(NUM_HEADS):
        w, pos = last[h].topk(TOP_ATTENDED)
        print(f"  head {h:2d}: " + ", ".join(f"{corpus.token_str(x[p])!r} {a:.2f}" for a, p in zip(w.tolist(), pos.tolist())))


code: ...'= dim\n\n    def forward(self, x):\n        return np'  (fused vs walkthrough on trained weights: 1.3e-06)
where the last token ' np' attends, per head:
  head  0: '.__' 0.55, '__(' 0.08, '\n\n   ' 0.03
  head  1: ' =' 0.88, 'ch' 0.02, 'import' 0.01
  head  2: ' =' 0.14, '):' 0.12, '):' 0.12
  head  3: ' np' 0.49, ' np' 0.49, ' forward' 0.01
  head  4: ' np' 0.48, ' np' 0.48, '):' 0.01
  head  5: ' np' 0.50, ' np' 0.50, '__(' 0.00
  head  6: ' return' 0.47, '\n\n' 0.09, ' numpy' 0.07
  head  7: ' forward' 0.18, 'import' 0.08, 'import' 0.08
  head  8: ' tor' 0.11, '.__' 0.08, ' __' 0.06
  head  9: ' return' 0.11, '(' 0.08, '(' 0.08
  head 10: '\n\n   ' 0.15, ' tor' 0.15, ' numpy' 0.10
  head 11: ' numpy' 0.49, 'import' 0.07, 'import' 0.07
  head 12: '()' 0.26, '\n\n' 0.23, ' En' 0.11
  head 13: ' __' 0.44, '(' 0.07, '(' 0.07
  head 14: ' np' 0.17, ' np' 0.17, ' forward' 0.07
  head 15: ' n' 0.15, ' dim' 0.06, ' dim' 0.06
  head 16: '__()' 0.24, 'Module' 0.10, ' return' 0.07
 

Each head's weights can be read as two (vocab, vocab) circuits: the QK circuit $W_E^T W_Q^T W_K W_E$ says which source tokens a destination token attends to, and the OV circuit $W_U W_O W_V W_E$ says how attending to a source changes the output logits. Chaining the two gives skip-trigrams `[source] … [destination] → [output]`.

For every frequent destination, the circuits propose candidates (top QK sources, then top OV outputs among tokens that can follow the destination), and a single counting pass over the corpus scores all of them at once. Each head's candidates are ranked by a smoothed lift, since raw ratios put one-off coincidences (1/1 = ×inf) on top.

In [6]:
MIN_COUNT = 1000         # destinations, sources and outputs must appear this often in the counted sample
ALL_SOURCES = 5          # top QK sources per destination
ALL_OUTPUTS = 5          # top OV outputs per (destination, source), among tokens that ever follow the destination
MIN_HITS = 5             # a trigram needs at least this many train occurrences to be ranked
SHOW_TOP = 10            # trigrams printed per head
LOOKBACK = 256           # s counts as "before d" if within this many tokens; over the full 1023 common tokens are
                         # nearly always present, which drags every lift toward ×1
COUNT_TOKENS = 50_000_000   # train tokens counted, split evenly across sources; memory per pass scales with this
DOMAIN = None            # rank by one source's counts only ("wiki", "code" or "math"); None pools all three

# QK circuit W_E W_Q W_K^T W_E^T: entry [d, s] is the attention score when destination d looks back at source s.
# only the rows for the given destinations are built, so we never materialize the full (vocab, vocab) matrix
@torch.no_grad()
def qk_rows(model, h, dests):
    return model.W_E[dests] @ model.W_Q[h] @ model.W_K[h].T @ model.W_E.T

# OV circuit W_E W_V W_O W_U: entry [s, o] is how much attending to source s raises the logit of output o
@torch.no_grad()
def ov_rows(model, h, srcs):
    return model.W_E[srcs] @ model.W_V[h] @ model.W_O[h] @ model.W_U

# paper's copying score sum(λ) / sum(|λ|) over the OV circuit's eigenvalues; the nonzero eigenvalues of
# W_E (W_V W_O W_U) equal those of (W_V W_O W_U) W_E, which is only (hidden, hidden)
@torch.no_grad()
def copying_score(model, h):
    eig = torch.linalg.eigvals(model.W_V[h] @ model.W_O[h] @ model.W_U @ model.W_E)
    return (eig.sum().real / eig.abs().sum()).item()

# untrained baseline: copying scores near its values mean the head's OV circuit hasn't learned anything
baseline = OneLayerAttentionOnlyTransformer(V, hidden_dim=H_DIM, num_heads=NUM_HEADS, head_dim=HEAD_DIM, device=device)
copying = [copying_score(toy_transformer, h) for h in range(NUM_HEADS)]
print("copying score per head (trained vs untrained):")
for h in range(NUM_HEADS):
    print(f"  head {h:2d}: {copying[h]:+.2f}  vs  {copying_score(baseline, h):+.2f}")
del baseline

# next_occ[u] = the next position holding the same token as u (len(ids) if none). a token at u is the most recent
# copy of itself in the window before t exactly when next_occ[u] >= t, which counts each source once per window
@torch.no_grad()
def next_occurrence(ids):
    order = torch.argsort(ids, stable=True)   # positions grouped by token, ascending within each token
    nxt = torch.full_like(ids, len(ids))
    same = ids[order[1:]] == ids[order[:-1]]
    nxt[order[:-1][same]] = order[1:][same]
    return nxt

# one pass over (u, t) with t - u in 1..LOOKBACK-1, i.e. s somewhere in the window before d. returns
#   pair_n[s * V + d]: positions t with ids[t] = d and s in the window before t (dense, V * V; accumulated into
#                      `pair_n` when given)
#   triple_k[i]: how many of those also have ids[t + 1] = o, for each sorted candidate key (s * V + d) * V + o
@torch.no_grad()
def count_skip_trigrams(ids, cand_keys, pair_n=None):
    nxt = next_occurrence(ids)
    if pair_n is None:
        pair_n = torch.zeros(V * V, dtype=torch.long, device=ids.device)
    triple_k = torch.zeros(len(cand_keys), dtype=torch.long, device=ids.device)
    for j in range(1, LOOKBACK):
        u = torch.arange(len(ids) - 1 - j, device=ids.device)   # t = u + j, and t + 1 must exist
        t = u + j
        keep = nxt[u] >= t
        u, t = u[keep], t[keep]
        pair = ids[u] * V + ids[t]
        pair_n += torch.bincount(pair, minlength=V * V)
        key = pair * V + ids[t + 1]
        idx = torch.searchsorted(cand_keys, key).clamp(max=len(cand_keys) - 1)
        hit = cand_keys[idx] == key
        triple_k += torch.bincount(idx[hit], minlength=len(cand_keys))
    return pair_n, triple_k

def fmt_lift(split, k, n, p_o_d):
    if n == 0:
        return f"{split} n=0"
    p = k / n
    lift = f"×{p / p_o_d:.1f}" if p_o_d > 0 else "×inf" if k else "×–"
    return f"{split} {k}/{n} = {p:.1%} vs {p_o_d:.2%} ({lift})"

tok = lambda i: repr(corpus.token_str(i))

# the counted sample: the first COUNT_TOKENS / 3 train tokens of each source, as int64
count_t = {s: ids[: COUNT_TOKENS // len(SOURCES)].long() for s, ids in train_t.items()}
val_count_t = {s: ids.long() for s, ids in val_t.items()}

# rare tokens' W_E rows are close to their random init, so their QK scores are noise; lone bytes of multi-byte
# characters print as "�" and aren't readable. neither is used as a destination, source or output, and
# neither is the document separator
counts = sum(torch.bincount(ids, minlength=V) for ids in count_t.values())
partial = torch.tensor(["�" in corpus.token_str(i) for i in range(V)], device=device)
excluded = (counts < MIN_COUNT) | partial
excluded[corpus.eot_id] = True
all_dests = torch.nonzero(~excluded).squeeze(1)

# first pass, with a dummy candidate, for which sources ever occur before d and which outputs ever follow it
pair_total = torch.zeros(V * V, dtype=torch.long, device=device)
follows = torch.zeros(V, V, dtype=torch.bool, device=device)
for ids in count_t.values():
    count_skip_trigrams(ids, torch.zeros(1, dtype=torch.long, device=device), pair_n=pair_total)
    follows |= count_bigrams(ids) > 0
seen_pair = pair_total.view(V, V) > 0          # [s, d]
del pair_total

# candidates come from the circuits and the corpus only scores them, since counting every (s, d, o) in the data
# is out of reach. the OV row depends only on s, so its top outputs are often tokens that can never follow d
# (the paper's "skip-trigram bugs"); restricting outputs to tokens seen after d skips those
cands = []   # (head, s, d, o) rows
for h in range(NUM_HEADS):
    for chunk in all_dests.split(512):
        qk = qk_rows(toy_transformer, h, chunk).masked_fill(excluded | ~seen_pair[:, chunk].T, float("-inf"))
        s_score, srcs = qk.topk(ALL_SOURCES, dim=-1)                               # (n_d, ALL_SOURCES)
        ok = s_score.isfinite()                                                    # a d with < ALL_SOURCES valid sources
        d_flat, s_flat = chunk[:, None].expand_as(srcs)[ok], srcs[ok]
        ov = ov_rows(toy_transformer, h, s_flat).masked_fill(excluded | ~follows[d_flat], float("-inf"))
        o_score, outs = ov.topk(ALL_OUTPUTS, dim=-1)
        rows = torch.stack([torch.full_like(outs, h), s_flat[:, None].expand_as(outs),
                            d_flat[:, None].expand_as(outs), outs], dim=-1)[o_score.isfinite()]
        cands.append(rows)
cands = torch.cat(cands)
keys = (cands[:, 1] * V + cands[:, 2]) * V + cands[:, 3]
cand_keys, inverse = torch.unique(keys, return_inverse=True)                        # sorted, as searchsorted needs
h_, s_, d_, o_ = cands.unbind(-1)
del seen_pair, follows

# per source and split, at every candidate: k hits, n windows with s before d, and the bigram counts for P(o | d).
# the dense (V, V) tables are gathered at the candidates and freed right away, one source at a time
stats = {}
for split, streams in (("train", count_t), ("val", val_count_t)):
    for s, ids in streams.items():
        pair_n, triple_k = count_skip_trigrams(ids, cand_keys)
        bigrams = count_bigrams(ids)
        stats[split, s] = dict(k=triple_k[inverse], n=pair_n[s_ * V + d_], bi=bigrams[d_, o_], d_total=bigrams.sum(dim=1)[d_])
        del pair_n, triple_k, bigrams

def pooled(split, field, sources):
    return sum(stats[split, s][field] for s in sources)

# lift = P(o | s…d) / P(o | d): above 1 means seeing s earlier makes o likelier after d, as the head predicts.
# raw lift puts one-off coincidences (1/1 = ×inf) on top; adding one expected and one observed hit damps small n
ranked = SOURCES if DOMAIN is None else (DOMAIN,)
k, n = pooled("train", "k", ranked), pooled("train", "n", ranked)
p_o_d = pooled("train", "bi", ranked) / pooled("train", "d_total", ranked).clamp(min=1)
smoothed_lift = (k + 1) / (n * p_o_d + 1)
vk, vn = pooled("val", "k", ranked), pooled("val", "n", ranked)
v_p_o_d = pooled("val", "bi", ranked) / pooled("val", "d_total", ranked).clamp(min=1)
# which source each trigram's train hits come from, whatever DOMAIN is
hits_by_source = torch.stack([stats["train", s]["k"] for s in SOURCES], dim=-1).float()
print(f"\n{len(cand_keys):,} candidate skip-trigrams across {NUM_HEADS} heads, {len(all_dests):,} destinations; "
      f"ranked on {'all sources' if DOMAIN is None else DOMAIN}")

for h in sorted(range(NUM_HEADS), key=lambda h: -copying[h]):
    sel = torch.nonzero((h_ == h) & (k >= MIN_HITS)).squeeze(1)
    sel = sel[smoothed_lift[sel].argsort(descending=True)[:SHOW_TOP]]
    print(f"\nhead {h} (copying score {copying[h]:+.2f}): [source] … [destination] → [output], top {len(sel)} by smoothed lift")
    for i in sel.tolist():
        share = hits_by_source[i] / hits_by_source[i].sum()
        main = share.argmax().item()
        checks = [fmt_lift("train", k[i].item(), n[i].item(), p_o_d[i].item()), fmt_lift("val", vk[i].item(), vn[i].item(), v_p_o_d[i].item())]
        print(f"  {tok(s_[i]):>14} … {tok(d_[i]):<14} → {tok(o_[i]):<14} [{SOURCES[main]} {share[main]:.0%}]  {' | '.join(checks)}")

copying score per head (trained vs untrained):
  head  0: +0.82  vs  +0.00
  head  1: +0.69  vs  +0.01
  head  2: +0.80  vs  -0.00
  head  3: +0.88  vs  +0.02
  head  4: +0.63  vs  -0.01
  head  5: +0.76  vs  -0.03
  head  6: +0.83  vs  -0.02
  head  7: +0.31  vs  -0.01
  head  8: +0.33  vs  -0.01
  head  9: +0.57  vs  +0.03
  head 10: +0.62  vs  +0.02
  head 11: +0.73  vs  +0.03
  head 12: +0.86  vs  +0.03
  head 13: +0.80  vs  +0.01
  head 14: +0.42  vs  -0.02
  head 15: +0.86  vs  +0.01
  head 16: +0.83  vs  -0.02
  head 17: +0.87  vs  -0.01
  head 18: +0.33  vs  +0.01
  head 19: +0.74  vs  -0.02
  head 20: +0.82  vs  +0.01
  head 21: +0.70  vs  +0.01
  head 22: +0.86  vs  +0.00
  head 23: +0.86  vs  +0.00
  head 24: +0.84  vs  -0.00
  head 25: +0.73  vs  +0.01
  head 26: +0.83  vs  -0.00
  head 27: +0.60  vs  +0.01
  head 28: +0.82  vs  -0.01
  head 29: +0.77  vs  -0.01
  head 30: +0.40  vs  -0.01
  head 31: +0.72  vs  +0.02

4,501,207 candidate skip-trigrams across 32 heads, 5,680